# Sakha GRPO Training Notebook

Train a hospital ward assistant using GRPO (Group Relative Policy Optimization) on the Sakha environment.

**Requirements**: GPU runtime (T4 is sufficient for 0.6B model)

**What this does**:
1. Installs Sakha environment and dependencies
2. Configures GRPO training with HF TRL
3. Runs training for N episodes
4. Plots reward curves and metrics

## 1. Setup - Install Dependencies

In [ ]:
import importlib.util, sys, os

# Idempotent setup: skip install if already available (e.g. after restart)
if importlib.util.find_spec("unsloth") is not None:
    print("Dependencies already installed. Skipping setup.")
    if os.path.exists("sakha"):
        %cd sakha
else:
    print("First run: installing dependencies...")
    !git clone -b main https://github.com/atharva-again/sakha.git
    %cd sakha
    !pip install uv -q
    !uv pip install --system -e ".[dev]"
    !uv pip install --system git+https://github.com/huggingface/transformers.git@main trl jmespath
    # Pin vllm==0.18.0 to avoid torch.compile crash in 0.19.x
    # See: https://github.com/unslothai/unsloth/issues/4841
    !uv pip install -qqq vllm==0.18.0 torchvision bitsandbytes xformers unsloth
    print("\n" + "="*60)
    print("INSTALL COMPLETE: Please restart your runtime now!")
    print("Colab: Runtime -> Restart session (Ctrl+M .)")
    print("Then re-run all cells starting from this one.")
    print("="*60)
    raise SystemExit("Restart required")

In [ ]:
# Optional: Mount Google Drive to cache HF models across sessions
# This saves ~5-10 min on every restart by reusing downloaded weights
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print(f"HF cache set to: {os.environ['HF_HOME']}")
except Exception:
    print("Google Drive not mounted. Models will cache in default ~/.cache/huggingface/")

## HF Token Setup

For Round 2 submission, use a T4 GPU (free Colab) for smoke tests and an A100 (HF credits) for full training.

Set your HuggingFace token in Colab secrets (left panel → 🔑) as  to access gated models.

## 2. Verify Environment

In [ ]:
import sys
sys.path.insert(0, "src")

from sakha.env import SakhaEnvironment
from sakha.models import SakhaAction, ActionType

# Quick smoke test
env = SakhaEnvironment(task="hard")
obs = env.reset()
print(f"Patients: {len(obs.ward_state.patients)}")
print(f"Pending tasks: {obs.pending_count}")
print(f"Step: {obs.ward_state.current_step}")

## 3. Configure Training

Adjust these parameters based on your GPU and time constraints:

In [ ]:
# Training configuration
MODEL = "Qwen/Qwen3-0.6B"          # Model to train (0.6B fits on T4)
TASK = "hard"                      # Task difficulty: easy | medium | hard
EPISODES = 200                       # Number of training episodes
MAX_STEPS = 96                       # Max steps per episode (96 = full 8hr shift)
SEED = 42                            # Random seed for reproducibility

# Unsloth config (set USE_UNSLOTH=True for 4-bit training on T4)
USE_UNSLOTH = True                   # Use Unsloth for memory-efficient training
LOAD_IN_4BIT = True                  # 4-bit quantization (critical for T4)

# GRPO specific
NUM_GENERATIONS = 4                  # Responses per prompt
LEARNING_RATE = 1e-5                 # Learning rate
MAX_COMPLETION_LENGTH = 512          # Max tokens per completion

print(f"Training config:")
print(f"  Model: {MODEL}")
print(f"  Task: {TASK}")
print(f"  Episodes: {EPISODES}")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Use Unsloth: {USE_UNSLOTH}")
print(f"  4-bit quant: {LOAD_IN_4BIT}")
print(f"  Estimated time: ~{EPISODES * 2} minutes on T4")

## 4. Define Environment Wrapper for TRL

In [ ]:
from sakha.env import SakhaEnvironment
from sakha.models import SakhaAction, ActionType

class SakhaEnvWrapper:
    """Environment wrapper for TRL's GRPOTrainer."""

    def __init__(self, task: str = "hard", seed: int | None = None):
        self.task = task
        self.seed = seed
        self.env = SakhaEnvironment(task=task)
        self.reward = 0.0
        self._action_type = ActionType
        self._episode_reward = 0.0
        self._episode_steps = 0

    def reset(self, **kwargs) -> str:
        """Reset the environment."""
        if self.seed is not None:
            import random
            random.seed(self.seed)
        obs = self.env.reset()
        self.reward = 0.0
        self._episode_reward = 0.0
        self._episode_steps = 0
        return self._format_observation(obs)

    def _format_observation(self, obs) -> str:
        """Format observation as string for tool result."""
        pending = obs.ward_state.pending_tasks[:5] if obs.ward_state.pending_tasks else []
        tasks_str = ", ".join(f"{t.required_action}({t.patient_id})" for t in pending) or "No pending tasks"
        return (
            f"Step {obs.ward_state.current_step}, "
            f"Patients: {len(obs.ward_state.patients)}, "
            f"Pending: {obs.pending_count}, "
            f"Tasks: {tasks_str}"
        )

    def _step(self, action) -> str:
        """Execute action and track metrics."""
        obs = self.env.step(action)
        self.reward = obs.reward or 0.0
        self._episode_reward += self.reward
        self._episode_steps += 1
        return obs

    def review_patient(self, patient_id: int) -> str:
        """Review patient at bed ID.

        Args:
            patient_id: The bed number of the patient to review.
        """
        action = SakhaAction(action_type=self._action_type.REVIEW_PATIENT, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def administer_medicine(self, patient_id: int) -> str:
        """Administer medication to patient.

        Args:
            patient_id: The bed number of the patient to administer medicine to.
        """
        action = SakhaAction(action_type=self._action_type.ADMINISTER_MEDICINE, patient_id=patient_id)
        obs = self._step(action)
        detail = obs.action_result.detail if obs.action_result else ""
        return f"{self._format_observation(obs)} | {detail}"

    def check_vitals(self, patient_id: int) -> str:
        """Check vitals for patient.

        Args:
            patient_id: The bed number of the patient to check vitals for.
        """
        action = SakhaAction(action_type=self._action_type.CHECK_VITALS, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def alert_doctor(self, patient_id: int) -> str:
        """Alert doctor for patient.

        Args:
            patient_id: The bed number of the patient to alert the doctor about.
        """
        action = SakhaAction(action_type=self._action_type.ALERT_DOCTOR, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def escalate(self, patient_id: int) -> str:
        """Escalate patient incident.

        Args:
            patient_id: The bed number of the patient to escalate.
        """
        action = SakhaAction(action_type=self._action_type.ESCALATE, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def update_chart(self, patient_id: int) -> str:
        """Update patient chart.

        Args:
            patient_id: The bed number of the patient whose chart to update.
        """
        action = SakhaAction(action_type=self._action_type.UPDATE_CHART, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def prepare_discharge(self, patient_id: int) -> str:
        """Prepare patient for discharge.

        Args:
            patient_id: The bed number of the patient to prepare for discharge.
        """
        action = SakhaAction(action_type=self._action_type.PREPARE_DISCHARGE, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def medication_round(self) -> str:
        """Complete medication round for all patients with due medications."""
        action = SakhaAction(action_type=self._action_type.MEDICATION_ROUND, patient_id=None)
        obs = self._step(action)
        detail = obs.action_result.detail if obs.action_result else ""
        return f"{self._format_observation(obs)} | {detail}"

    def ward_sweep(self) -> str:
        """Complete ward sweep and coordination check."""
        action = SakhaAction(action_type=self._action_type.WARD_SWEEP, patient_id=None)
        obs = self._step(action)
        return self._format_observation(obs)

    def noop(self) -> str:
        """Take no action (pass time)."""
        action = SakhaAction(action_type=self._action_type.NOOP, patient_id=None)
        obs = self._step(action)
        return self._format_observation(obs)

    def get_metrics(self):
        """Return accumulated episode metrics."""
        return {
            "episode_reward": self._episode_reward,
            "episode_steps": self._episode_steps,
        }

def reward_func(environments, **kwargs):
    """Reward function for GRPO.

    Must use **kwargs to handle extra parameters from TRL.
    """
    return [getattr(env, "reward", 0.0) for env in environments]

## 5. Load Model with Unsloth (Optional)

In [ ]:
# Load model with Unsloth for memory-efficient training on T4
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=MAX_COMPLETION_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        fast_inference=True,  # Set False if vLLM crashes (slower but stable)
        gpu_memory_utilization=0.7,
    )

    # Add LoRA adapters for parameter-efficient training
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=32,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    print(f"Unsloth model loaded: {MODEL}")
    print(f"4-bit quantization: {LOAD_IN_4BIT}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
else:
    model = MODEL  # Use string model name in GRPOTrainer
    tokenizer = None
    print(f"Vanilla model: {MODEL}")

## 6. Create Dataset and Configure GRPO

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

def create_prompt(task: str, episode_id: int = 0):
    system_msg = (
        "You are a hospital ward assistant managing patients. "
        "Available actions: review_patient(patient_id), check_vitals(patient_id), "
        "administer_medicine(patient_id), alert_doctor(patient_id), escalate(patient_id), "
        "update_chart(patient_id), prepare_discharge(patient_id), medication_round(), "
        "ward_sweep(), noop(). "
        f"Task difficulty: {task}. "
        f"Episode: {episode_id}. "
        "Choose the best action based on pending tasks and patient needs."
    )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": "Begin your shift. Review the ward and decide your first action."},
    ]


# Build dataset
prompts = [create_prompt(TASK, i) for i in range(EPISODES)]
dataset = Dataset.from_dict({"prompt": prompts})

print(f"Dataset size: {len(dataset)} prompts")

In [ ]:
# GRPO configuration optimized for T4
grpo_config = GRPOConfig(
    num_train_epochs=1,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    per_device_train_batch_size=2,       # Reduced for T4 safety
    gradient_accumulation_steps=2,       # Effective batch = 4
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    report_to=[],  # Disable wandb for Colab
    output_dir="./grpo_output",
    seed=SEED,
    remove_unused_columns=False,
    use_vllm=USE_UNSLOTH,
    vllm_gpu_memory_utilization=0.3 if USE_UNSLOTH else None,
)

print("GRPO Config:")
print(f"  Generations: {NUM_GENERATIONS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max completion: {MAX_COMPLETION_LENGTH}")
print(f"  Batch size: 2 (accumulation: 2)")
print(f"  Save checkpoints: every 50 steps")
print(f"  Use vLLM: {USE_UNSLOTH}")

## 12. Quick Heuristic Evaluation

In [ ]:
import os
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

# Build trainer kwargs
trainer_kwargs = {
    "train_dataset": dataset,
    "reward_funcs": reward_func,
    "args": grpo_config,
    "environment_factory": lambda: SakhaEnvWrapper(task=TASK, seed=SEED),
}

if USE_UNSLOTH:
    trainer_kwargs["model"] = model
    if tokenizer is not None:
        trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["model"] = MODEL

trainer = GRPOTrainer(**trainer_kwargs)

print(f"Starting training: {EPISODES} episodes, task={TASK}")
print(f"Using Unsloth: {USE_UNSLOTH}")
print("This will take a while... grab a coffee ☕")

trainer.train()

## 7. Pre-Training Evaluation — Base LLM

Evaluate the base Qwen3-0.6B model (zero-shot, no training) on the same seeds.
This establishes the 'before' baseline that proves RL training improved the model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sakha.env import SakhaEnvironment
from sakha.graders import score_easy_task, score_medium_task, score_hard_task
from sakha.models import SakhaAction, ActionType
import re

# Load base model (no training)
print("Loading base Qwen3-0.6B for zero-shot evaluation...")
base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-0.6B", trust_remote_code=True, padding_side="left"
)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

# Action name mapping
ACTION_NAME_MAP = {
    "review_patient": ActionType.REVIEW_PATIENT,
    "check_vitals": ActionType.CHECK_VITALS,
    "administer_medicine": ActionType.ADMINISTER_MEDICINE,
    "alert_doctor": ActionType.ALERT_DOCTOR,
    "escalate": ActionType.ESCALATE,
    "update_chart": ActionType.UPDATE_CHART,
    "prepare_discharge": ActionType.PREPARE_DISCHARGE,
    "medication_round": ActionType.MEDICATION_ROUND,
    "ward_sweep": ActionType.WARD_SWEEP,
    "noop": ActionType.NOOP,
}

SYSTEM_PROMPT = (
    "You are a hospital ward assistant managing patients. "
    "Available actions: review_patient(patient_id), check_vitals(patient_id), "
    "administer_medicine(patient_id), alert_doctor(patient_id), escalate(patient_id), "
    "update_chart(patient_id), prepare_discharge(patient_id), medication_round(), "
    "ward_sweep(), noop(). "
    "Choose the best action based on pending tasks and patient needs."
)

def build_eval_prompt(obs):
    pending = obs.ward_state.pending_tasks[:5] if obs.ward_state.pending_tasks else []
    tasks_str = ", ".join(f"{t.required_action}({t.patient_id})" for t in pending) or "No pending tasks"
    return (
        f"Step {obs.ward_state.current_step}. "
        f"Patients: {len(obs.ward_state.patients)}. "
        f"Pending: {obs.pending_count}. "
        f"Tasks: {tasks_str}. "
        f"What action do you take?"
    )

def parse_action_response(response):
    response = response.strip().lower()
    match = re.search(r"(\w+)\s*\(\s*(\d+)?\s*\)", response)
    if match:
        name = match.group(1)
        patient_id = int(match.group(2)) if match.group(2) else None
        if name in ACTION_NAME_MAP:
            return SakhaAction(action_type=ACTION_NAME_MAP[name], patient_id=patient_id)
    return SakhaAction(action_type=ActionType.NOOP, patient_id=None)

def run_llm_eval(task, model, tokenizer, seeds, max_steps, device="cuda"):
    TASK_GRADERS = {"easy": score_easy_task, "medium": score_medium_task, "hard": score_hard_task}
    PATIENT_COUNTS = {"easy": 5, "medium": 8, "hard": 18}
    pc = PATIENT_COUNTS[task]
    grader = TASK_GRADERS[task]
    episodes = []
    for seed in seeds:
        env = SakhaEnvironment(patient_count=pc, task=task)
        obs = env.reset(seed=seed)
        trajectory = [obs]
        step_rewards = []
        for step in range(max_steps):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_eval_prompt(obs)},
            ]
            inputs = tokenizer.apply_chat_template(
                messages, return_tensors="pt", add_generation_prompt=True
            ).to(device)
            with torch.no_grad():
                outputs = model.generate(
                    inputs, max_new_tokens=64, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            action = parse_action_response(response)
            obs = env.step(action)
            trajectory.append(obs)
            step_rewards.append(obs.reward)
            if obs.done:
                break
        grader_score = grader(trajectory)
        metrics = env.episode_metrics
        episodes.append({
            "seed": seed, "grader_score": grader_score,
            "total_reward": sum(step_rewards),
            "critical_incidents_resolved": metrics.critical_incidents_resolved,
            "critical_incidents_missed": metrics.critical_incidents_missed,
            "overdue_tasks": metrics.overdue_tasks,
            "noop_steps": metrics.noop_steps,
            "discharges_prepared": metrics.discharges_prepared,
        })
    def mean(key):
        return sum(e[key] for e in episodes) / len(episodes)
    return {
        "task": task, "policy": "base_llm", "episodes": episodes,
        "summary": {
            "mean_grader_score": mean("grader_score"),
            "mean_total_reward": mean("total_reward"),
            "mean_critical_incidents_resolved": mean("critical_incidents_resolved"),
            "mean_critical_incidents_missed": mean("critical_incidents_missed"),
            "mean_overdue_tasks": mean("overdue_tasks"),
            "mean_noop_steps": mean("noop_steps"),
            "mean_discharges_prepared": mean("discharges_prepared"),
        }
    }

EVAL_SEEDS = list(range(42, 52))  # 10 seeds for speed on T4
base_results = run_llm_eval(TASK, base_model, base_tokenizer, EVAL_SEEDS, MAX_STEPS)
print(f"Base LLM mean grader score: {base_results['summary']['mean_grader_score']:.4f}")
print(f"Base LLM mean total reward: {base_results['summary']['mean_total_reward']:.4f}")

# Free VRAM
del base_model
del base_tokenizer
torch.cuda.empty_cache()

## 8. Post-Training Evaluation — Trained LLM

Load the GRPO checkpoint and evaluate on the same seeds.

In [ ]:
from peft import PeftModel

# Discover latest checkpoint
from pathlib import Path
ckpts = sorted(Path("./grpo_output").glob("checkpoint-*"))
if not ckpts:
    print("No checkpoints found. Training may still be running or failed.")
else:
    CHECKPOINT_PATH = str(ckpts[-1])
    print(f"Loading trained checkpoint: {CHECKPOINT_PATH}")

    trained_tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-0.6B", trust_remote_code=True, padding_side="left"
    )
    if trained_tokenizer.pad_token is None:
        trained_tokenizer.pad_token = trained_tokenizer.eos_token

    base_for_trained = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B",
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    trained_model = PeftModel.from_pretrained(base_for_trained, CHECKPOINT_PATH)
    trained_model = trained_model.merge_and_unload()
    trained_model.eval()

    trained_results = run_llm_eval(TASK, trained_model, trained_tokenizer, EVAL_SEEDS, MAX_STEPS)
    print(f"Trained mean grader score: {trained_results['summary']['mean_grader_score']:.4f}")
    print(f"Trained mean total reward: {trained_results['summary']['mean_total_reward']:.4f}")

    del trained_model
    del base_for_trained
    del trained_tokenizer
    torch.cuda.empty_cache()

## 9. Base vs Trained Comparison

Side-by-side metrics and bar chart.

In [ ]:
import matplotlib.pyplot as plt

b = base_results["summary"]
t = trained_results["summary"]

print("\n## Base LLM vs Trained LLM Comparison\n")
print("| Metric | Base LLM | Trained | Delta |")
print("|--------|----------|---------|-------|")
print(f"| Grader Score | {b['mean_grader_score']:.4f} | {t['mean_grader_score']:.4f} | {t['mean_grader_score'] - b['mean_grader_score']:+.4f} |")
print(f"| Total Reward | {b['mean_total_reward']:.4f} | {t['mean_total_reward']:.4f} | {t['mean_total_reward'] - b['mean_total_reward']:+.4f} |")
print(f"| Critical Resolved | {b['mean_critical_incidents_resolved']:.2f} | {t['mean_critical_incidents_resolved']:.2f} | {t['mean_critical_incidents_resolved'] - b['mean_critical_incidents_resolved']:+.2f} |")
print(f"| Critical Missed | {b['mean_critical_incidents_missed']:.2f} | {t['mean_critical_incidents_missed']:.2f} | {t['mean_critical_incidents_missed'] - b['mean_critical_incidents_missed']:+.2f} |")
print(f"| Overdue Tasks | {b['mean_overdue_tasks']:.2f} | {t['mean_overdue_tasks']:.2f} | {t['mean_overdue_tasks'] - b['mean_overdue_tasks']:+.2f} |")
print(f"| NOOP Steps | {b['mean_noop_steps']:.2f} | {t['mean_noop_steps']:.2f} | {t['mean_noop_steps'] - b['mean_noop_steps']:+.2f} |")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["Grader Score", "Total Reward", "Critical Resolved", "Overdue Tasks"]
base_vals = [b['mean_grader_score'], b['mean_total_reward'], b['mean_critical_incidents_resolved'], b['mean_overdue_tasks']]
trained_vals = [t['mean_grader_score'], t['mean_total_reward'], t['mean_critical_incidents_resolved'], t['mean_overdue_tasks']]
x = range(len(metrics))
width = 0.35
ax.bar([i - width/2 for i in x], base_vals, width, label='Base LLM', color='#e74c3c')
ax.bar([i + width/2 for i in x], trained_vals, width, label='Trained LLM', color='#2ecc71')
ax.set_ylabel('Value')
ax.set_title('Base vs Trained LLM Performance')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./grpo_output/base_vs_trained.png", dpi=150)
plt.show()
print("Plot saved to ./grpo_output/base_vs_trained.png")

## 13. Download Results

In [ ]:
# Zip results for download
!zip -r grpo_results.zip grpo_output/

from google.colab import files
files.download("grpo_results.zip")

---

## Summary

This notebook demonstrated GRPO training on the Sakha hospital ward environment.

**Key takeaways**:
- Sakha provides a realistic hospital workflow simulation with 96-step episodes
- GRPO with Unsloth 4-bit training fits on a free Colab T4 GPU
- Compare base LLM vs trained LLM on identical seeds to measure RL improvement
- Small models (0.6B) can learn basic patterns, but larger models (4B+) show better reasoning

**Next steps**:
- Try larger models (Qwen3-4B) for better performance
- Increase episodes to 500+ for meaningful convergence
- Add eval split to track generalization
- Experiment with different reward shaping